# 00 · EDA and Feature Checks helper: the stored artifacts, for reading the code

**What it is for.** A reference to keep open while you read the EDA and Feature Checks code. It
loads what the two stages stored and shows, for each artifact, its grain, its columns and a few
telling rows, so you can see what a function receives and returns.

**It is not a stage.** It only reads files: no Spark session, no stage run, no audit-log entry.
It shows the latest run of each stage for the config and entity of the setup cell.

**Before you start.** Run `02_eda` step 1 and `03_feature_checks` step 1 for the same config and
entity.

**How it is laid out**, in the order the code builds the artifacts:
1. the data definition's folder;
2. EDA: `eda.json`, then its tables: the counts table (`cells`) first, since every statistic is
   read from it, then `ranking`, `client_constant`, `correlation`, every other table, and the views the
   code review follows (section 8a);
3. the feature ledger;
4. Feature Checks: `feature_checks.json`, then `findings`, `unavailable`, `per_period`,
   `clusters` and `proposed_decisions`;
5. the decision audit log.

Each section says what the artifact is, its grain (one row per …), its key columns, which
functions write it and which read it (module and function under `src/core_modelling/`, so you
can jump to the code), and what to look at.

**Docs:** `docs/components/eda.md` and `docs/components/feature_checks.md` (Interface →
Artifacts: every key and column) · `docs/components/feature_ledger.md` (Columns).

## Setup
The same names as `03_feature_checks`. `resolve_config` replays the decision audit log onto the
config, as every stage does first; the artifacts sit in the data definition's folder that
`feature_dir` returns. Three functions read them, each checking the format version first:
`read_stage` (`artifacts.py`) a stage's json and every table it lists, `read_ledger`
(`ledger.py`) the ledger, refused when written for another data definition or by another run
than the stage folders' (`expect`: each stage's `run_token`, which the ledger's sidecar
records too).

`NUMERIC` and `CATEGORICAL` are two candidates followed through the tables: change them to follow
others.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

import core_modelling
from core_modelling.artifacts import read_stage
from core_modelling.config import load_config
from core_modelling.config.audit_log import resolve_config, show_decisions
from core_modelling.environment import feature_dir, load_environment
from core_modelling.ledger import read_ledger

# The use case and where the framework runs, as absolute paths (as in 03_feature_checks).
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"  # or conf/base/td_acquisition.yml
ENTITY = "entity_a"
ENVIRONMENT = str(REPO / "conf/local/environment.yml")  # or None: $CORE_MODELLING_ENVIRONMENT
NUMERIC = "outflow_casa_vol_0_5_months_avg"  # any numeric candidate
CATEGORICAL = "external_assets_band"  # any categorical candidate

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

env = load_environment(ENVIRONMENT)
resolved = resolve_config(load_config(CONFIG), ENTITY, env)
folder = feature_dir(env, resolved, ENTITY)
eda_meta, eda_tables = read_stage(feature_dir(env, resolved, ENTITY, "eda"), "eda")
fc_folder = feature_dir(env, resolved, ENTITY, "feature_checks")
fc_meta, fc_tables = read_stage(fc_folder, "feature_checks")
runs = {"eda": eda_meta["run_token"], "feature_checks": fc_meta["run_token"]}
ledger = read_ledger(folder, resolved.housekeeping["data_definition_hash"], expect=runs)
snapshot = eda_meta["snapshot_period"]
print(resolved.use_case, resolved.task, ENTITY, folder)

## 1. The data definition's folder
`<roots.data>/<use case>/04_feature/<data-definition hash>/` holds:
- `decisions/`: the decision audit log, one json file per entry (section 16);
- `eda/` and `feature_checks/`: each stage's `<stage>.json`, `tables/*.parquet`, `log.json` (the
  run's log entries), `resolved_config.yml` and `report.html`;
- `ledger.parquet` with its sidecar `_ledger.json`, which holds one run token per stage: the
  one that stage wrote into its json, so a ledger and a stage folder from different runs are
  refused.

Written by `write_stage` (`artifacts.py`: the folder whole, through a temporary sibling) and
`save_ledger` (`ledger.py`). A stage saves the ledger first and its folder last, so
`<stage>.json` is the commit marker. An EDA run removes `feature_checks/`: its judgement would be
stale.

In [ ]:
display({"folder": folder, "eda": feature_dir(env, resolved, ENTITY, "eda"),
         "feature_checks": feature_dir(env, resolved, ENTITY, "feature_checks")})
display({"eda tables": eda_meta["tables"], "feature_checks tables": fc_meta["tables"]})

## 2. `eda.json`: EDA's meta
One dict. Built by `_stage_meta` (`eda/run.py`), its head by `stage_identity` (`artifacts.py`);
`write_stage` adds the stamp (`schema_version`,
`stage`, `framework_version`, `written_at`, `tables`). The keys, by group:
- **Identity:** `entity`, `config_hash`, `data_definition_hash`, `resolved_input_hash`,
  `hash_version`, `replayed_to` (the last audit-log entry the config was replayed to).
- **Settings:** `settings` (the values EDA ran with at the paths of
  `SETTINGS_PATHS_BY_STAGE["eda"]` in `artifacts.py`: `features.kinds`, `strata`, `eda`,
  `time.validation_period`, `decision`, `evaluation.slices`) and `settings_digest`, their hash;
  `run_token`, this run's name, which the ledger's sidecar records too.
- **Periods:** `snapshot_period`; `compared_periods` (`window`, labelled; `gap` and `latest`,
  compared without labels).
- **Population:** `population`, the counts per period (label counts null outside the window);
  `clients`, the window's labelled rows and effective clients (a client labelled in several
  periods counts about once), which scale a support.
- **Features:** `candidates`, `kinds`, `kind_overrides`, `unsupported`, `lineage`,
  `window_levels` (a categorical's levels in the window), `over_ceiling`, `key_features`,
  `represented_by`.
- **Report:** `headline`, `warnings`, `slices`, `snapshot_base`, `collect`, `timings`.

**Read by** `read_upstream` (`artifacts.py`: `read_stage`, then `check_upstream`, which refuses
EDA's artifacts when the data definition or the settings digest differ from the current resolved
config, naming the changed path) and `check_inputs` (`feature_checks/inputs.py`: `compared_periods`, `snapshot_period`,
`over_ceiling`, `clients`).

In [ ]:
display(
    pd.DataFrame(
        {
            "type": {key: type(value).__name__ for key, value in eda_meta.items()},
            "value": {key: str(value)[:100] for key, value in eda_meta.items()},
        }
    )
)
display(eda_meta["compared_periods"], pd.DataFrame(eda_meta["population"]), eda_meta["clients"])

## 3. EDA's tables: the overview
`read_stage` returns every table EDA wrote, by name, in the order `eda.json`'s `tables` lists
them. The next four sections explain four of the five tables Feature Checks reads (`EDA_TABLES` in
`feature_checks/inputs.py`); section 8 lists the fifth, `concordance_per_period`, and shows every
other one, including a table added after this
notebook was written.

In [ ]:
pd.DataFrame(
    {
        name: {"rows": len(table), "columns": ", ".join(table.columns)}
        for name, table in eda_tables.items()
    }
).T

## 4. `cells`: the counts table
**Every per-feature statistic of EDA and Feature Checks is a sum over these cells.** One row per
`feature × period × lagged × fold × bin × target_bin`, over the eligible rows of every compared
period:
- `n` the rows in the cell; `sum_y` and `sum_y2` the target's sum and sum of squares (null where
  the target is unknown).
- `bin`: a numeric's value range (`< e`, `[e, e')`, `≥ e`, or `= v` for a single value) or a
  categorical's level; `bin_order` its position; `is_null` the null bin, always its own.
- `target_bin`: the target's value (binary) or its quantile bin (regression); null where the
  label is unknown, which is every row of the gap and latest periods.
- `lagged`: the previous period's value against this period's target, for the window periods
  only; the window's first period has no earlier row, so its lagged cells are all the null bin.
- `fold`: a categorical's client fold in the window periods, for the cross-fitted level encoding;
  null for a numeric and outside the window.

**Written by** `count_cells` and `count_lagged_cells` (`dataset/counts.py`, the two Spark passes;
the edges from `bin_edges`, the levels from `window_levels`), called from `_count_tables` in
`eda/run.py`. **Read by** `scope_statistics` (`statistics/scope.py`, called per feature by `feature_statistics`
in `eda/ranking.py`) through `statistics/cells.py` (`pool_periods` sums the cells of the periods
asked for), the per-period concordance and the slices, and in Feature Checks by `check_inputs`
(`feature_checks/inputs.py`) and the steps (`feature_checks/steps.py`, `measures.py`).

**Look at:** the cells per period (no lagged cells outside the window), the numeric's snapshot
cells by bin and target bin, the categorical's cells with their folds, the numeric's lagged rows
per window period (what `lagged_concordance` in `feature_checks/measures.py` reads), and the last
table: the
ledger's `missing_share` is the null cells' share of the snapshot's rows.

In [ ]:
cells = eda_tables["cells"]
display(cells.dtypes.to_frame("dtype").T)
display(cells.groupby(["period", "lagged"]).size().unstack("lagged"))
numeric = cells.query("feature == @NUMERIC and period == @snapshot and not lagged")
display(
    numeric.fillna({"target_bin": "unknown"}).pivot_table(
        index=["bin_order", "bin"], columns="target_bin", values="n", aggfunc="sum"
    )
)
display(cells.query("feature == @CATEGORICAL and period == @snapshot and not lagged").head(12))
display(cells.query("feature == @NUMERIC and lagged").groupby("period")["n"].sum().to_frame("lagged rows").T)
null_share = numeric.loc[numeric["is_null"], "n"].sum() / numeric["n"].sum()
display(
    pd.Series(
        {
            "null cells' share of the snapshot's rows": null_share,
            "ledger missing_share": ledger.set_index("feature").at[NUMERIC, "missing_share"],
        }
    )
)

## 5. `ranking`: every statistic per candidate
One row per candidate EDA read (lineage and unsupported columns have none), in rank order:
- the snapshot's statistics and their `*_window` twins (the window periods' cells pooled):
  `concordance`, `missing_share`, `null_concordance`, `zero_share`, `dominant_share`;
- `concordance_se`, `direction`, `concordance_best_order` and `non_monotone`, and
  `concordance_best_order_window` (the best order on the window, numerics only);
- `raw_sign` and `raw_sign_window` (+1 or −1): the direction every per-period and lagged
  concordance is read in, so a flipped feature reads below 0.5;
- `unavailable`: why a concordance cannot exist ('' when it can);
- `ranking_value`, `rank`, `key_feature` (`top_n`, `listed` or '') and `represented_by` (the key
  feature it was skipped for, as a near-duplicate).

**Written by** `feature_statistics` → `rank_features` → `select_key_features` (`eda/ranking.py`,
called from `_rank` in `eda/run.py`); each feature's concordance, SE and direction come from
`scope_statistics`, `feature_se` and `concordance_direction` (`statistics/scope.py`). **Read by** `_ledger` (`eda/run.py`: the ledger's EDA
columns are copied from it), the report's ranking view, and in Feature Checks by `check_inputs`
(`feature_checks/inputs.py`: `raw_sign_window`, `concordance_window`,
`concordance_best_order_window`, `unavailable`; the higher of the two window concordances is what
`leakage.too_strong` reads).

**Look at:** the top rows (a concordance near 1 is usually the target under another name, which
Feature Checks' `leakage.too_strong` flags) and the two candidates' rows.

In [ ]:
ranking = eda_tables["ranking"]
display(ranking.dtypes.to_frame("dtype").T)
display(ranking.head(10))
display(ranking.set_index("feature").loc[[NUMERIC, CATEGORICAL]].T)

## 6. `client_constant`: values that never change within a client
One row per candidate read: `constant_share`, the share of clients with at least two non-null
values in the window whose value never changes, out of `clients` such clients.

**Written by** `client_constant_shares` (`dataset/counts.py`, the second Spark pass). **Read by**
`check_time_alignment` (`feature_checks/steps.py`, the `time_alignment.client_constant` flag);
the ledger's `constant_share` is this column.

**Look at:** a share near 1 for a value that should move (a balance, a count) suggests one date's
value stamped onto every period; a static attribute (a nationality) is constant by nature.

In [ ]:
client_constant = eda_tables["client_constant"]
display(client_constant.sort_values("constant_share", ascending=False).head(10))
display(client_constant.query("feature in [@NUMERIC, @CATEGORICAL]"))

## 7. `correlation`: the association between candidates
One row per pair of candidates read, each pair once (`a`, `b`): `statistic` by `method`,
`spearman` between two numerics, `cramers_v` between two categoricals, `eta` between a numeric
and a categorical; `rows` the rows on which both are non-null. Computed on one seeded sample of
the snapshot's eligible rows (`pairwise_sample`).

**Written by** `association_table` (`statistics/association.py`, called from `_rank` in
`eda/run.py`). **Read by** `select_key_features` (a key feature's near-duplicates are skipped:
`represented_by`), the report's correlation view, and `check_redundancy`
(`feature_checks/steps.py`), which cuts the clusters with `cluster_features`.

**Look at:** the methods' counts, and the numeric's strongest partners: those above
`redundancy.correlation_max` end in its cluster (section 14).

In [ ]:
correlation = eda_tables["correlation"]
display(correlation["method"].value_counts())
partners = correlation.query("a == @NUMERIC or b == @NUMERIC")
display(partners.sort_values("statistic", key=abs, ascending=False).head(8))

## 8. EDA's other tables: the report's views
Each feeds a view of the report and of `02_eda` through `build_views` (`eda/report.py`); Feature
Checks reads none of them except `concordance_per_period`. The writers
(`eda/population.py` unless named):
- `population`: the counts per period, as in `eda.json` (`build_frames`, `dataset/frames.py`).
- `funnel`: the rows per window period after each eligibility step (`funnel_per_period`).
- `sources`: the share of the spine's rows each feature source matched, per compared period
  (`sources_per_period`).
- `target_per_period`: the target's rate (mean) per window period, with its interval and
  support (`target_per_period`).
- `strata`: the target by each stratum's levels, then each key feature by stratum (`feature`
  filled) (`strata_tables`, `strata_pivots` in `eda/deep_dives.py`, `strata_frame`).
- `reference_groups`: the excluded clients against the eligible ones (`reference_groups`).
- `slices` and `capture_curve`: the key features at the decision's slices (`slice_tables`,
  `eda/slices.py`; the cut itself is `capture_table` and `capture_curve` in
  `statistics/capture.py`).
- `profile`: each candidate's value range or largest levels (`value_profile`, `eda/profile.py`).
- `concordance_per_period`: every candidate's concordance per window period, in the window's
  direction, `insufficient` below `strata.min_events` (`concordance_per_period` over
  `per_period_concordance`, `eda/deep_dives.py`); Feature Checks reads it for
  `concordance_unstable` and its stability chart.
- The deep dives, one block per key feature (`eda/deep_dives.py`): `bins` (a numeric's merged
  bins) or `levels` (a categorical's levels), both by `group_table`; `quartiles` by target class;
  `per_period`, the key features' rows of `concordance_per_period`; `target_quartiles` for a
  regression target.

**Look at:** each table's first rows for its grain; a table not listed here is newer than this
notebook and shows too.

In [ ]:
explained = {"cells", "ranking", "client_constant", "correlation"}
for name, table in eda_tables.items():
    if name not in explained:
        display(Markdown(f"#### `{name}`: {len(table)} rows"), table.head(5))

## 8a. The views the code review follows
Four of EDA's other tables, cut to the rows that show what a function returns; the review
guides in `worknotes/` point here by number.
1. **`concordance_per_period`** for the two candidates: one row per window period, the
   concordance read in the window's direction, its SE and `insufficient`
   (`per_period_concordance`, `eda/deep_dives.py`, one `scope_statistics` per period). Feature
   Checks' `concordance_unstable` and its Cochran's Q read these rows (`_heterogeneity`,
   `_period_ses` in `feature_checks/steps.py`).
2. **`slices`** of the first key feature: the list cut at each evaluation slice, with its
   estimate, lift and capture (`capture_table`, `statistics/capture.py`), and five points of its
   **`capture_curve`**.
3. **`bins`** of the first key feature that has them: its deep-dive groups (`group_table`,
   `eda/deep_dives.py`), beside its **`quartiles`** by target class. A key numeric with few
   distinct values (a flag, a count) is shown by value in `levels` instead.
4. **`target_per_period`** and **`population`**: the target's rate per window period with its
   interval, and the counts per compared period (label counts null outside the window).

**Look at:** a period whose concordance sits apart from the others; the capture at the smallest
slice against the slice's share; the lift from the first group to the last.

In [ ]:
per_period_concordance = eda_tables["concordance_per_period"]
display(per_period_concordance.query("feature in [@NUMERIC, @CATEGORICAL]"))
key = next(iter(eda_meta["key_features"]))
display(eda_tables["slices"].query("feature == @key"))
display(eda_tables["capture_curve"].query("feature == @key and share in [0.01, 0.05, 0.1, 0.25, 0.5]"))
binned = eda_tables["bins"]["feature"].iloc[0]
display(eda_tables["bins"].query("feature == @binned"))
display(eda_tables["quartiles"].query("feature == @binned"))
display(eda_tables["target_per_period"], eda_tables["population"])

## 9. The feature ledger
**One row per candidate of the data definition**, lineage and unsupported columns included. Its
stored columns come in three groups, in this order:
- **identity** (`feature` … `unsupported`): `create_ledger` (`ledger.py`), from `_ledger` in
  `eda/run.py`;
- **EDA's** (`levels` … `represented_by`): the ranking's columns, joined by `add_columns(...,
  "eda")`;
- **Feature Checks'** (`flags` … `partners`): each step's columns (`StepResult.columns`), joined
  by `_ledger_columns` (`feature_checks/run.py`) with `add_columns(..., "feature_checks")`; a
  skipped step's are null, and an EDA rerun drops them.

**Written by** `save_ledger` (with `_ledger.json`: the format version, the data definition and
`runs`, one run token per stage whose columns it holds), **read by** `read_ledger`. Feature Checks
reads its candidates (`read_candidates`: neither lineage nor unsupported) and EDA's statistics
from it (`check_inputs`, `feature_checks/inputs.py`); from Training on, `resolve_features`
(`ledger.py`) gives the feature set: candidates − lineage − unsupported − excluded − held.

**The ledger stores no decision.** `held`, `kept`, `excluded` and `verdict` are derived whenever
a view is drawn (`render_decisions`, `ledger.py`) from the resolved config, where the replay
writes the open flags' features into `features.held` and the person's decisions into
`features.excluded` and `features.kept` (the last table).

**Look at:** a flagged row (`flags` lists `step.finding` names), a clean row (empty lists) and a
lineage row (only its identity filled).

In [ ]:
display(ledger.dtypes.to_frame("dtype").T)
flagged, clean = ledger["flags"].str.len() > 0, ledger["flags"].str.len() == 0
lineage = ledger["lineage"]
examples = [*ledger.index[flagged][:1], *ledger.index[clean][:1], *ledger.index[lineage][:1]]
display(ledger.loc[examples].set_index("feature").T)
display(
    pd.Series(
        {
            "held": resolved.features.held,
            "excluded": resolved.features.excluded,
            "kept": resolved.features.kept,
        }
    ).to_frame("features in the resolved config")
)

## 10. `feature_checks.json`: Feature Checks' meta
One dict. Built by `_stage_meta` (`feature_checks/run.py`) plus `write_stage`'s stamp. Beside
the identity keys, as in `eda.json`:
- `settings` and `settings_digest`: EDA's paths plus `feature_checks`, as the run read them
  (Training checks them); `run_token`, this run's name;
- `snapshot_period`, `compared_periods`, `steps_run`, `steps_skipped`;
- `findings`: per finding of the steps that ran, its `kind`, `count` and `features` (the values
  stay in the `findings` table);
- `held`: the held features when this run ended;
- `entries_written`: the audit-log entries this run wrote (a rerun on unchanged data writes only
  its `run` entry);
- `eda_headline`, `timings`.

**Look at:** `held` is the state at the end of this run, while the resolved config's
`features.held` (section 9) includes every entry recorded since: the two differ after a later
`keep`, `exclude` or `revert`.

In [ ]:
display(
    pd.DataFrame(
        {
            "type": {key: type(value).__name__ for key, value in fc_meta.items()},
            "value": {key: str(value)[:100] for key, value in fc_meta.items()},
        }
    )
)
display(pd.DataFrame(fc_meta["findings"]).T, pd.DataFrame(fc_meta["entries_written"]))

## 11. `findings`: what each step raised
One row per feature and finding raised: `feature`, `step`, `finding` (the full name
`step.finding`), `kind` (`flag` holds the feature out of training until a person decides, `note`
never affects training), `value` (the statistic the finding reads) against
`threshold` (its config leaf), `text` (the finding's fixed text, also its `flag` entry's reason)
and `detail` (what the row adds: a minority's effect, a pair's partners, the periods tested,
and for a finding with a standard error its support, SE and "not significant at this support"
within 2 SE of the threshold).

**Written by** the five `check_<step>` functions (`feature_checks/steps.py`, run in `STEPS`
order; names, kinds and texts in `FINDINGS`, the rows built by `finding_rows`, both in
`findings.py`), stacked by `stack_parts`. **Read by**
`flag_entries` (`decisions.py`: a `flag` entry per finding whose feature set changed since its
last entry), `proposed_decisions`, `held_summary`, the ledger's `flags` and `notes`, and the
report's step sections.

**Look at:** the features per finding, the two candidates' findings, and one row of each finding.

In [ ]:
findings = fc_tables["findings"]
display(findings.groupby(["step", "finding", "kind"], sort=False).size().to_frame("features"))
display(findings.query("feature in [@NUMERIC, @CATEGORICAL]"))
display(findings.groupby("finding", sort=False).head(1))

## 12. `unavailable`: what a step could not compute
One row per feature and finding a step could not compute, with the `reason` ("not available: …"):
no value can exist, such as no events or a one-period window. No check skips for low support: a
thin result is computed and its detail says whether it is significant. Not a finding: the feature
is neither flagged nor cleared for it. Decided before computing, never by catching an error.

**Written by** each `check_<step>` (`StepResult.unavailable`); **read by** the report's "could not
test" tables, one per step.

In [ ]:
unavailable = fc_tables["unavailable"]
display(unavailable.groupby("finding").size().to_frame("features"), unavailable.head(10))

## 13. `per_period`: each candidate across the compared periods, label-free
One row per candidate read and compared period: `group` (`window`, `gap` or `latest`), `rows`,
`missing_share` and `psi` against the snapshot period. No label: the concordance per window
period is EDA's `concordance_per_period` (section 8).

**Written by** `per_period_table` (`feature_checks/measures.py`) from EDA's cells, inside
`check_inputs` (`feature_checks/inputs.py`). **Read by** the report's per-period PSI table; `check_stability`'s
`psi_window` and `psi_latest` come from `period_psi` in the same module.

**Look at:** the numeric across the periods: its PSI against the snapshot in the gap and latest
periods, the data production will score.

In [ ]:
per_period = fc_tables["per_period"]
display(per_period.dtypes.to_frame("dtype").T)
display(per_period.query("feature == @NUMERIC"))

## 14. `clusters`: groups of near-duplicates
One row per member of a cluster: `cluster` (its number), `feature`, `kind`, `concordance`,
`association_with_best` (with the cluster's strongest member), `explained_by_others` (the
within-cluster R² on ranks; null when not computable, with the reason in `unavailable`).

**Written by** `check_redundancy` (`feature_checks/steps.py`, its `StepResult.tables`), which
cuts the clusters from EDA's `correlation` at `redundancy.correlation_max` with
`cluster_features` (`statistics/association.py`). **Read by** the report's cluster summary and the
ledger's `cluster`; Training's grouped importance will read the ledger's column.

**Look at:** the first clusters, and the numeric's cluster when it has one.

In [ ]:
clusters = fc_tables["clusters"]
display(clusters.head(10))
display(clusters.loc[clusters["cluster"].isin(clusters.query("feature == @NUMERIC")["cluster"])])

## 15. `proposed_decisions`: the review
One row per flag finding with open features: `finding`, `step`, `features` and `values` (lists,
one value per feature; None where this run did not raise it), `threshold`, `text`, and the two
`record_decision` calls a person can copy, `keep_call` and `exclude_call`.

**Written by** `proposed_decisions` (`feature_checks/decisions.py`) from the findings and the open
flags (`open_flags`, `config/replay.py`). **Read by** the report's review section and
`03_feature_checks` step 9.

**Look at:** one row's features beside their values, and its two calls.

In [ ]:
proposed = fc_tables["proposed_decisions"]
display(proposed.drop(columns=["keep_call", "exclude_call"]))
display(proposed[["finding", "keep_call", "exclude_call"]].head(1).T)

## 16. The decision audit log
One row per entry, in order: `seq`, `time`, `by` (a stage or a person), `stage`, `op` (`flag`,
`run`, `keep`, `exclude`, `include`, `set`, `revert`), `what`, `reason`, `evidence`. On disk, one
json file per entry under `decisions/`.

**Written by** `record_stage_entry` (`config/audit_log.py`: Feature Checks' `flag` entries, an
empty one closing a finding, and its `run` entry) and `record_decision` (a person's entries).
**Read by** `resolve_config`, which replays the entries onto the config (`config/replay.py`:
`decisions_in_force` gives `features.excluded` and `features.kept`, `open_flags` and
`held_features` give `features.held`) at the start of every stage, and here by
`show_decisions`.

**Look at:** the `flag` entries a run wrote, the person's `keep` and `exclude` entries after it,
and how a `revert` undoes the entries after its target.

In [ ]:
decisions = show_decisions(resolved, ENTITY, env)
display(decisions["op"].value_counts().to_frame("entries"), decisions)